# Day 13 — Morfolojik İşlemler, Kenar ve Çizgi Tespiti
## Erozyon, Genleşme, Canny Kenar ve Probabilistic Hough Dönüşümü ile Dokuma Hizalama

> **Aşama:** Faz 2 — Bilgisayarlı Görü (Day 09–15)
> **Resmi Staj Defteri Konusu:** Morfolojik İşlemler, Kenar ve Çizgi Tespiti (Yaprak 25 & 26)

### 1. Problem
Halı üretiminde çözgü ve atkı ipliklerinin hizalanması, bordür paralelliği ve dokuma sırasındaki mikro iplik kopuklukları çıplak gözle incelenemeyecek kadar hızlıdır. İkili (binary) görüntü maskelerindeki parazitleri temizlemek ve çizgisel iplik yapılarını otomatik tespit etmek gerekir.

### 2. Why the Problem Matters
Matematiksel morfoloji (açma/kapama) ile ikili maskelerdeki izole gürültü noktaları yok edilirken iplik delikleri kapatılır. Canny kenar ve Hough çizgi dönüşümü ise tezgâh boyunca ipliklerin açısal eğriliklerini tespit eder.

### 3. Engineering Concepts
- **Yapılandırıcı Eleman (Structuring Element)**: Şekil ve boyutu belirleyen morfolojik çekirdek matrisi.
- **Açma (Opening)**: Erozyon ardından genleşme (küçük izole parazitleri yok eder).
- **Kapama (Closing)**: Genleşme ardından erozyon (küçük delik ve çatlakları doldurur).
- **Canny Kenar Tespiti**: Gradyan büyüklüğü ve yönü, non-maximum suppression ve çift eşikleme.
- **Probabilistic Hough Transform (HoughLinesP)**: Parametrik kutupsal uzayda çizgisel doğru tespiti.

In [ ]:
# 4. Library / API Investigation
import cv2
import numpy as np
from day13.mini_project.src.morphology_lines import MorphologyEdgeEngine

print("Morfoloji ve Çizgi Tespit Motoru Yüklendi.")

In [ ]:
# 5. Minimal Implementation
# İplik ve bordür çizgileri içeren sentetik görsel
canvas = np.zeros((200, 200), dtype=np.uint8)
canvas[40:160, 40:160] = 255
# Yapay parazit ekleme
canvas[20, 20] = 255
canvas[180, 180] = 255

opened = MorphologyEdgeEngine.apply_morphology(canvas, "opening", kernel_size=3)
edges = MorphologyEdgeEngine.detect_canny_edges(opened, 50, 150)
lines = MorphologyEdgeEngine.detect_lines_hough(edges, threshold=30, min_line_length=40, max_line_gap=5)
print(f"Açma Sonrası Temizlendi | Bulunan Kenar Çizgisi Sayısı: {len(lines)}")

In [ ]:
# 6. Experiment: İlk Doğruların Koordinatları
for idx, (x1, y1, x2, y2) in enumerate(lines[:4]):
    length = np.hypot(x2 - x1, y2 - y1)
    print(f"Çizgi {idx+1}: ({x1}, {y1}) -> ({x2}, {y2}) | Uzunluk: {length:.1f}px")

In [ ]:
# 7. Visualization: Morfoloji ve Tespit Edilen Doğrular
import matplotlib.pyplot as plt

line_img = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
for x1, y1, x2, y2 in lines:
    cv2.line(line_img, (x1, y1), (x2, y2), (0, 255, 0), 2)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(edges, cmap="gray")
axes[0].set_title("Canny Kenar Haritası")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(line_img, cv2.COLOR_BGR2RGB))
axes[1].set_title("Hough Çizgi Tespiti")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert len(lines) >= 4, "Kare desende en az 4 ana kenar çizgisi bulunmalıdır!"
assert opened[20, 20] == 0, "İzole gürültü noktası açma işlemiyle silinmiş olmalıdır."
print("Morfoloji ve kenar doğrulamaları başarılı.")

In [ ]:
# 9. Failure Cases: Geçersiz morfoloji işlemi
try:
    MorphologyEdgeEngine.apply_morphology(canvas, "unknown_op")
except ValueError as e:
    print("Beklenen hata yakalandı:", e)

### 10. Conclusions
Morfolojik filtreler ve Probabilistic Hough Çizgi dönüşümü uygulanmış, dokuma desenlerindeki yapısal doğrular ve parazitsiz kenar haritaları başarıyla elde edilmiştir.